# 02 — Supervised Fine-Tuning (SFT)

**Módulo:** `src/llm_rlhf/sft.py` → `SupervisedFineTuner`

SFT es *aprendizaje por imitación* para modelos de lenguaje. Le mostramos al modelo miles de demostraciones de alta calidad con la forma `(instrucción, respuesta)` y minimizamos la pérdida estándar de entropía cruzada sobre los tokens de la respuesta. La función de pérdida no es nueva — lo nuevo son los *datos*.

## Setup (Colab o entorno nuevo)

Esta primera celda es **idempotente**: si `llm_rlhf` ya está instalado, no hace nada. En Colab, la primera vez clona el repositorio y lo instala en modo editable.

El repositorio apuntado es [emiliomunozai/LLM_RLHF](https://github.com/emiliomunozai/LLM_RLHF).

In [ ]:
# --- Colab / fresh-environment setup ---------------------------------
# No-op if llm_rlhf is already importable (e.g. local uv environment).
import os, sys, subprocess

REPO_URL = "https://github.com/emiliomunozai/LLM_RLHF.git"
REPO_DIR = "LLM_RLHF"

try:
    import llm_rlhf  # noqa: F401
except ImportError:
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    print("Installed llm_rlhf. If torch/transformers were upgraded, restart the runtime now.")

## Qué enseña SFT, y qué no

**Enseña:** el *formato* de una interacción útil.

- `User: ... Assistant: ...` es una plantilla con significado.
- Los turnos del asistente son *respuestas*, no listas de más preguntas.
- Las respuestas terminan (no divagan hasta tocar el límite de tokens).

**No enseña:** la diferencia entre una respuesta buena y una mala.

Un dataset de 15.000 ejemplos no puede cubrir todo tipo de pregunta, y aunque pudiera, solo puede mostrar *una* respuesta buena por prompt. El modelo aprende "así se ven las respuestas", pero no "esta respuesta es mejor que aquella". Eso es lo que harán las etapas siguientes.

> **Por qué es el primer paso:** RLHF necesita un modelo que ya produzca respuestas del *formato* correcto para poder optimizar su *calidad*. Sin SFT, el modelo de recompensa estaría puntuando texto que ni siquiera intenta ser una respuesta.

## LoRA — por qué no entrenamos todos los pesos

OPT-350M tiene ~350 millones de parámetros. Actualizarlos todos por paso de gradiente:

- requeriría ~2× la memoria del modelo solo para el estado de Adam,
- sobreescribiría el conocimiento del preentrenamiento si el learning rate no se ajusta con mucho cuidado,
- y produciría un checkpoint de 700 MB por cada fine-tune.

LoRA (Low-Rank Adaptation) congela los pesos originales e inserta dos matrices pequeñas $A \in \mathbb{R}^{d \times r}$, $B \in \mathbb{R}^{r \times d}$ en cada proyección de atención. Solo entrenamos $A$ y $B$, con $r = 16$. La actualización efectiva es el producto de bajo rango $A B$, sumado a los pesos originales en inferencia.

Resultado: entrenamos ~0.5% de los parámetros y el adaptador guardado pesa unos pocos MB. Eso es lo que hace a LoRA *compartible*: puedes publicar un adaptador de unos MB en lugar de un modelo de cientos de MB.

In [ ]:
from llm_rlhf import PretrainedLLM
from llm_rlhf.sft import SupervisedFineTuner, SFTConfig
from llm_rlhf.config import load_toml

llm = PretrainedLLM()
cfg = SFTConfig(**load_toml('../configs/sft.toml'))
sft = SupervisedFineTuner(llm, cfg)

## El formato de Dolly-15k

`databricks/databricks-dolly-15k` son 15.000 pares prompt/respuesta escritos a mano por empleados de Databricks, en categorías como QA abierto, brainstorming, clasificación y resumen. Colapsamos cada fila en una única cadena (la plantilla usa llaves para los campos, ver `format_instruction()` en `sft.py`).

Cada etapa posterior usa *exactamente* esta plantilla para que el modelo no tenga que aprender un formato nuevo. Mira las primeras líneas decodificadas en la celda siguiente para verlo concretamente.

## Inspeccionando un ejemplo

Antes de entrenar, miremos exactamente lo que recibe el modelo: tokens y máscara de pérdida. Los tokens marcados con `labels = -100` se ignoran al calcular la pérdida — esto evita que el padding domine la señal.

In [ ]:
# Look at one *decoded* formatted example to see exactly what the model sees.
sft.prepare_data()

example = sft.tokenized_dataset['train'][0]
decoded = sft.tokenizer.decode(example['input_ids'], skip_special_tokens=False)
n_real    = sum(1 for t in example['labels'] if t != -100)
n_masked  = sum(1 for t in example['labels'] if t == -100)

print(decoded[:400] + (' ...' if len(decoded) > 400 else ''))
print()
print(f'sequence length    : {len(example["input_ids"])}')
print(f'tokens used in loss: {n_real}')
print(f'masked (-100)      : {n_masked}')

## Pérdida de LM causal, con un truco

El objetivo de entrenamiento es la **entropía cruzada habitual sobre predicción del siguiente token** — la misma del preentrenamiento. En cada posición $t$ de la secuencia el modelo emite logits sobre todo el vocabulario; comparamos contra el token *real* en la posición $t+1$ y minimizamos:

$$\mathcal{L}_{\text{SFT}} = - \frac{1}{|\mathcal{R}|} \sum_{t \in \mathcal{R}} \log \pi_\theta(y_{t+1} \mid y_{\leq t})$$

donde $\mathcal{R}$ es el conjunto de posiciones *no enmascaradas*. No hay nada nuevo en el cálculo — lo nuevo es **qué cuenta como token "correcto"**: ya no Wikipedia, sino la respuesta humana cuidadosamente escrita.

**El truco** es enmascarar los tokens que no queremos que dominen el gradiente, fijando sus etiquetas en `-100`. La función de pérdida de Hugging Face ignora automáticamente esa posición. Aquí enmascaramos solo el *padding*, así que el modelo aprende tanto a producir la respuesta del asistente como a copiar la pregunta del usuario que precede a `Assistant:`. Algunas implementaciones van más allá y enmascaran también los tokens del prompt para que la pérdida solo vea la respuesta — un compromiso entre velocidad de aprendizaje y coherencia de formato.

In [ ]:
# Attach LoRA adapters (only q_proj and v_proj by default — the attention query/value projections).
sft.setup_peft()

## Visualizando dónde viven los parámetros entrenables

La afirmación "LoRA entrena solo el 0.5% de los parámetros" es fácil de decir y difícil de creer hasta que la ves. La siguiente celda agrupa los parámetros por módulo y los apila: gris = congelados, naranja = entrenables.

Vas a ver dos cosas:

1. La mayoría de las barras son enteramente grises (capas no tocadas por LoRA).
2. Las pocas barras con una franja naranja arriba son las proyecciones de atención (`q_proj`, `v_proj`). Esa franja diminuta es todo lo que se entrena.

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# Group parameter counts by the top-level module they belong to.
trainable = defaultdict(int)
frozen    = defaultdict(int)
for name, p in sft.model.named_parameters():
    bucket = name.split('.')[-2] if '.' in name else name
    (trainable if p.requires_grad else frozen)[bucket] += p.numel()

buckets = sorted(trainable.keys(), key=lambda k: -trainable[k])
tr = [trainable[b] / 1e6 for b in buckets]
fr = [frozen[b]    / 1e6 for b in buckets]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(buckets, fr, label='frozen', color='#cccccc')
ax.bar(buckets, tr, bottom=fr, label='trainable (LoRA)', color='#ff7f50')
ax.set_ylabel('parameters (M)')
ax.set_title('Where the trainable parameters live')
ax.tick_params(axis='x', rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

total_tr = sum(trainable.values())
total_fr = sum(frozen.values())
print(f'Total trainable: {total_tr/1e6:.2f}M ({total_tr/(total_tr+total_fr):.2%} of all params)')

## Entrenamiento (descomenta para ejecutar de verdad)

Un epoch sobre OPT-350M tarda ~15 minutos en una GPU de consumo. Lo omitimos de este notebook para que la explicación se lea rápido; descoméntalo cuando estés listo.

In [ ]:
# adapter_path = sft.train()
# llm.load_adapter(adapter_path)
# print(sft.generate('Explain quantum computing to a 10-year-old:'))

## Después del entrenamiento: re-evalúa el caso canónico

Aquí entra en escena la idea de **encadenar etapas**. Si corriste la celda anterior, el adaptador LoRA quedó guardado en `checkpoints/sft/adapter`. La siguiente celda es *idempotente*:

- Si el adaptador existe en disco, lo engancha al modelo y genera.
- Si no existe (todavía no entrenaste), genera con el modelo base — exactamente como en `01_pretrained`.

Esa convención se repite en todas las notebooks aguas abajo (`03`, `04`, `05`, `06`), así que cada una funciona en aislamiento *y* aprovecha los checkpoints anteriores cuando existen.

In [ ]:
import os
from llm_rlhf import CANONICAL_PROMPT, EVAL_PROMPTS
from llm_rlhf.eval import SFT_ADAPTER_PATH

# Idempotent: load the SFT adapter if it exists, otherwise stay with the base model.
status = "base (SFT adapter not found — train first to see the difference)"
if os.path.exists(os.path.join("..", SFT_ADAPTER_PATH)):
    llm.load_adapter(os.path.join("..", SFT_ADAPTER_PATH))
    status = "SFT-aligned"

print(f"Generating with: {status}\n")
for p in EVAL_PROMPTS:
    print(f"PROMPT:     {p}")
    print(f"COMPLETION: {sft.generate(p) if status == 'SFT-aligned' else llm.generate(p)}")
    print()

### Progresión esperada — `CANONICAL_PROMPT`

Aunque OPT-350M nunca va a ser ChatGPT, la diferencia tras un epoch de SFT sobre Dolly-15k es **estructural**, no de conocimiento. Esto es lo que se ve realmente al correr el pipeline en este repo (RTX 5070 Ti, ~4 minutos de entrenamiento):

| Etapa | Salida real sobre `"Explain quantum computing to a 10-year-old."` |
|---|---|
| **Base** (notebook 01) | `What's quantum computing?` (el modelo continúa el texto como si fuera el inicio de un artículo y rebota la pregunta — no responde) |
| **SFT** (este notebook) | `quantum computing is a technology that uses a quantum field to create a quantum computer, which can be used to create quantum computing. The term quantum computing refers to the technology that uses…` (responde en formato declarativo, aunque el contenido es tautológico y reciclado) |

Las dos cosas a observar:

1. **El formato cambió.** Antes: una pregunta de retorno. Ahora: una afirmación que *intenta definir* el concepto. Esto es lo único que SFT le pidió aprender.
2. **El contenido aún es pobre.** El modelo se queda atrapado en repetición ("a quantum computer ... quantum computing ... quantum"). Eso es lo que produce OPT-350M cuando se le pide un tema sobre el que no tiene conocimiento real. *Ese* problema no lo resuelve SFT — es trabajo para las etapas de RLHF (notebooks 04–06) o, más realista, para escalar a un modelo más grande.

> **Por qué esto importa para `03_reward_model`:** el modelo SFT ya genera *múltiples* respuestas distintas para el mismo prompt (cambia el seed o la temperatura). El modelo de recompensa aprenderá a *ordenar* esas respuestas. Sin SFT, todas las respuestas serían continuaciones tipo "What's quantum computing?" y no hay nada útil que ordenar.

## Ejercicio: el coste de LoRA según su configuración

La hiperparametrización de LoRA tiene dos perillas principales:

- **`r`** — el rango. Más alto significa más capacidad expresiva, pero también más parámetros entrenables.
- **`target_modules`** — qué proyecciones envolver. Lo mínimo viable es `q_proj, v_proj`; algunos trabajos envuelven las cuatro proyecciones de atención (más `o_proj`/`out_proj`).

La siguiente celda construye el adaptador para varias combinaciones *sin entrenar* — solo cuenta cuántos parámetros serían entrenables. Antes de correrla, intenta predecir:

- ¿Doblar `r` duplica los parámetros entrenables?
- ¿Pasar de 2 a 4 módulos objetivo los duplica?
- ¿Cuál es el "presupuesto" típico que aceptarías a cambio de mejor adaptación?

In [ ]:
# Exercise: how does the trainable-parameter count change with different LoRA configurations?
# Try a few combinations *without retraining* — just count what would be trainable.
from peft import LoraConfig, TaskType, get_peft_model

configs_to_try = [
    dict(r=4,  target=('q_proj', 'v_proj')),
    dict(r=16, target=('q_proj', 'v_proj')),
    dict(r=16, target=('q_proj', 'k_proj', 'v_proj', 'out_proj')),
    dict(r=64, target=('q_proj', 'v_proj')),
]

base = PretrainedLLM()
for c in configs_to_try:
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=c['r'],
        lora_alpha=c['r'] * 2,
        target_modules=list(c['target']),
        bias='none',
    )
    peft_model = get_peft_model(base.model, lora)
    n_train = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in peft_model.parameters())
    print(f"r={c['r']:>3}, targets={c['target']!s:<55} -> {n_train/1e6:5.2f}M trainable ({n_train/n_total:.2%})")
    base.model = peft_model.get_base_model()

## Siguiente: `03_reward_model.ipynb`

Ahora tenemos un modelo que *puede* seguir instrucciones. Para que *quiera* dar respuestas útiles, necesitamos una señal de qué significa "útil".